# Sales Prediction
**Task 5 - OASIS INFOBYTE Data Science Internship**

Predict product sales based on advertising spend across TV, Radio, and Newspaper channels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Load the Dataset

In [ ]:
df = pd.read_csv('advertising.csv', index_col=0)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 2. Exploratory Data Analysis

In [ ]:
sns.pairplot(df)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
features = ['TV', 'Radio', 'Newspaper']
for ax, feat in zip(axes, features):
    ax.scatter(df[feat], df['Sales'], alpha=0.6)
    ax.set_xlabel(feat)
    ax.set_ylabel('Sales')
    ax.set_title(f'{feat} vs Sales')
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

In [ ]:
X = df.drop(columns=['Sales'])
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')

---
## 4. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso Regression': Lasso(alpha=0.1),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'R2 Score': r2, 'RMSE': rmse, 'MAE': mae}
    
    print(f'\n{"="*40}')
    print(f'{name}')
    print(f'R2 Score: {r2:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE: {mae:.4f}')

In [ ]:
results_df = pd.DataFrame(results).T
results_df

In [ ]:
plt.figure(figsize=(10, 5))
bars = plt.bar(results_df.index, results_df['R2 Score'], color=['#2E86AB', '#A23B72', '#F18F01', '#4CAF50'])
plt.ylabel('R2 Score')
plt.title('Model Comparison - Sales Prediction')
plt.ylim(0, 1)
for bar, val in zip(bars, results_df['R2 Score']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold')
plt.show()

---
## 5. Make a Prediction

In [ ]:
best_model_name = results_df['R2 Score'].idxmax()
best_model = models[best_model_name]
print(f'Best model: {best_model_name} (R2: {results_df.loc[best_model_name, "R2 Score"]:.4f})')

sample = X_test_scaled[0:1]
pred = best_model.predict(sample)
actual = y_test.iloc[0]
print(f'Predicted Sales: {pred[0]:.2f}')
print(f'Actual Sales:    {actual:.2f}')